In [ ]:
import numpy as np
import uuid
from PIL import Image
from tqdm import tqdm 
from collections import Counter
import re

import os
import sys
import random
from tqdm import tqdm 
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:

path_to_hovernet = "/media/nas/MP/segmentation/hovernet/"
PNG_Paddy_7 = "PNG_Paddy07-2024/"

# Search for all subfolders (classes) in "PNG_Paddy07-2024/"
def find_subfolders(base_dir):
        subfolders = []
        for root, dirs, files in os.walk(base_dir): 
            for dir in dirs:
                subfolders.append(os.path.join(root, dir))
        return subfolders

all_subfolders = find_subfolders(path_to_hovernet+PNG_Paddy_7)
len(all_subfolders), all_subfolders[:]

In [ ]:
os.path.basename(all_subfolders[0])

In [ ]:
# Find all npz files for each class 
def find_npz_files(base_dirs):
    npz_dict = {}

    for base_dir in base_dirs:
        folder_name = os.path.basename(base_dir)
        for root, dirs, files in os.walk(base_dir):  # Recursively walk through subdirectories
            for file in files:
                if file.endswith('.npz'):  # Check if the file ends with .npz
                    npz_dict.setdefault(folder_name, []).append(os.path.join(root, file))

    return npz_dict

all_npz_dict = find_npz_files(all_subfolders)

In [ ]:
npz_counts_per_folder = {key: len(value) for key, value in all_npz_dict.items()}
print(npz_counts_per_folder)

In [ ]:
lower_bound = min(npz_counts_per_folder.values())
upper_bound = max(npz_counts_per_folder.values())
lower_bound, upper_bound

In [ ]:
len(all_npz_dict)

In [ ]:
# Create a dictionary with the npz files as values for each class
breed_1121_dict = {key: all_npz_dict[key] for key in npz_counts_per_folder.keys() if '1121' in key}
breed_1509_dict = {key: all_npz_dict[key] for key in npz_counts_per_folder.keys() if '1509' in key}
other_breeds_dict = {key: all_npz_dict[key] for key in npz_counts_per_folder.keys() if '1121' not in key and '1509' not in key}

# Sample the npz files for 1121 and 1051 class
max_n_samples = min(len(list(breed_1121_dict.values())[0]), len(list(breed_1509_dict.values())[0]))
sampled_breed_1121 = {key: random.sample(value, max_n_samples) for key, value in breed_1121_dict.items()} # sample untill the upperbound
sampled_breed_1509 = {key: random.sample(value, max_n_samples) for key, value in breed_1509_dict.items()}

# Sample the npz files for other classes
max_other_breeds_samples = round(max_n_samples / len(other_breeds_dict.keys())) # max samples for other classes
sampled_other_breeds = {key: random.sample(value, max_other_breeds_samples) for key, value in other_breeds_dict.items()} 

In [ ]:
sampled_breed_1509

In [20]:
sampled_other_breeds['PUSA_1718']

['/media/nas/MP/segmentation/hovernet/PNG_Paddy07-2024/PUSA_1718/PUSA_PADDY 1718_HARYANA_ 08.07.2024_20240708_0006.npz',
 '/media/nas/MP/segmentation/hovernet/PNG_Paddy07-2024/PUSA_1718/PUSA_PADDY 1718_HARYANA_ 08.07.2024_20240708_0012.npz',
 '/media/nas/MP/segmentation/hovernet/PNG_Paddy07-2024/PUSA_1718/PUSA_PADDY 1718_HARYANA_ 08.07.2024_20240708_0007.npz',
 '/media/nas/MP/segmentation/hovernet/PNG_Paddy07-2024/PUSA_1718/PUSA_PADDY 1718_HARYANA_ 08.07.2024_20240708_0008.npz',
 '/media/nas/MP/segmentation/hovernet/PNG_Paddy07-2024/PUSA_1718/PUSA_PADDY 1718_HARYANA_ 08.07.2024_20240708_0005.npz']

In [38]:
# Merge all sampled dictionaries of 1121 and 1509
all_sampled = {**sampled_breed_1121, **sampled_breed_1509, **sampled_other_breeds}

# Flatten the dictionary into a list of (Class, File Path)
data = [(class_name, file_path) for class_name, files in all_sampled.items() for file_path in files]
df_samples = pd.DataFrame(data, columns=["Class", "File Path"])

print(df_samples.shape)
print(df_samples.shape)

(71, 2)
(71, 2)


In [39]:
Counter(df_samples['Class'])

Counter({'PUSA_PADDY_1121_HARYANA_8-7-2024': 23,
         'PUSA_PADDY-1509_HARYANA_3-7-24': 23,
         'PUSA_1718': 5,
         'PUSA_SHARBATI': 5,
         'PUSA_SUGANDHA_HARYANA 3-7-24': 5,
         'PUSA HARYANA': 5,
         'PUSA_PADDY_1401_HARYANA 3-7-24': 5})

In [59]:
# First, split into train + temp (validation + test) using stratified split
train_df, test_df = train_test_split(df_samples, test_size=0.15, stratify=df_samples['Class'],random_state=42)

# Now, split temp into validation and test sets using stratified split
train_df, val_df = train_test_split(train_df, test_size=0.15, random_state=42)

# Reset indexes
train_df.reset_index(drop=True, inplace=True)
val_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)


In [60]:
Counter(train_df['Class']), Counter(val_df['Class']) , Counter(test_df['Class'])

(Counter({'PUSA_PADDY_1121_HARYANA_8-7-2024': 18,
          'PUSA_PADDY-1509_HARYANA_3-7-24': 18,
          'PUSA HARYANA': 4,
          'PUSA_SHARBATI': 4,
          'PUSA_1718': 3,
          'PUSA_PADDY_1401_HARYANA 3-7-24': 3,
          'PUSA_SUGANDHA_HARYANA 3-7-24': 1}),
 Counter({'PUSA_SUGANDHA_HARYANA 3-7-24': 3,
          'PUSA_PADDY-1509_HARYANA_3-7-24': 2,
          'PUSA_PADDY_1121_HARYANA_8-7-2024': 2,
          'PUSA_PADDY_1401_HARYANA 3-7-24': 1,
          'PUSA_1718': 1}),
 Counter({'PUSA_PADDY_1121_HARYANA_8-7-2024': 3,
          'PUSA_PADDY-1509_HARYANA_3-7-24': 3,
          'PUSA_1718': 1,
          'PUSA_SUGANDHA_HARYANA 3-7-24': 1,
          'PUSA_SHARBATI': 1,
          'PUSA HARYANA': 1,
          'PUSA_PADDY_1401_HARYANA 3-7-24': 1}))

In [61]:
track = "track_3_class"
os.makedirs(track, exist_ok=True)
train_df.to_csv(f"{track}/train3class.csv", index=False)
test_df.to_csv(f"{track}/test3class.csv", index=False)
val_df.to_csv(f"{track}/val3class.csv", index=False)

In [62]:
PNG_Paddy_7 = "PNG_Paddy07-2024-3Class/"
os.makedirs(f"{PNG_Paddy_7}/train", exist_ok=True)
os.makedirs(f"{PNG_Paddy_7}/test", exist_ok=True)
os.makedirs(f"{PNG_Paddy_7}/val", exist_ok=True)

In [63]:
def create_npz_dataset(df):
    x = []
    y_keys = []
    # Extract columns
    class_names = df["Class"].values
    file_paths = df["File Path"].values

    # Process images
    x = []
    y_keys = []

    for file_path, class_name in tqdm(zip(file_paths, class_names), desc="Processing images", total=len(file_paths)):
        images = np.load(file_path)['kernel_pics'].astype('uint8')
        y = [class_name] * images.shape[0]
        x.extend(images)
        y_keys.extend(y)

    x = np.array(x)
    y_keys = np.array(y_keys)

    return x, y_keys

In [64]:
X_train, Y_train = create_npz_dataset(train_df)
X_val, Y_val = create_npz_dataset(val_df)
X_test, Y_test = create_npz_dataset(test_df)

Processing images: 100%|██████████| 11/11 [00:19<00:00,  1.79s/it]


In [66]:
np.savez_compressed(f"{PNG_Paddy_7}/train/train_data.npz",x_train=X_train, y_train = Y_train)
np.savez_compressed(f"{PNG_Paddy_7}/val/val_data.npz",x_val=X_val, y_val=Y_val)
np.savez_compressed(f"{PNG_Paddy_7}/test/test_data.npz",x_test=X_test, y_test=Y_test)

In [ ]:
train_data = np.load("train_data.npz")